<a href="https://colab.research.google.com/github/elianafonseca/GEO01007/blob/main/SIG_OSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DEPARTAMENTO DE GEOGRAFIA - UFRGS

SISTEMAS DE INFORMAÇÕES GEOGRÁFICAS

Profa. ELIANA LIMA DA FONSECA (eliana.fonseca@ufrgs.br)

Assunto: OPEN STREET MAP (OSM)

ANTES DE INICIAR: Salvar uma copia no seu proprio drive. Usar o menu lateral para "montar o google drive"

In [ ]:
# importa as bibliotecas já instaladas
# a biblioteca pandas já está previamente instalada no colab
import pandas as pd

#instala a biblioteca geopandas (só precisa instalar uma vez)
!pip install geopandas
# importa a biblioteca geopandas
import geopandas as gpd

### Carregando o Shapefile da Restinga

Nesta seção, carregamos o arquivo shapefile (`.shp`) que define o limite do bairro Restinga. Utilizamos a biblioteca `geopandas` para ler o shapefile e armazená-lo em um GeoDataFrame, que é uma estrutura de dados semelhante ao `pandas.DataFrame`, mas com uma coluna adicional para dados geoespaciais (`geometry`).

Certifique-se de que o caminho `shape_path` esteja correto para onde seu arquivo `restinga_wgs84.shp` está armazenado no Google Drive.

In [ ]:
# informa o caminho do shape
shape_path = "/content/drive/MyDrive/shapes/restinga_wgs84.shp"


# associa o shape à variavel
restinga = gpd.read_file(shape_path)

### Extraindo a Rede de Ruas com OSMnx

A biblioteca `osmnx` é uma ferramenta poderosa para trabalhar com dados OpenStreetMap (OSM) em Python. Aqui, extraímos a rede de ruas (graph) para a localização "Restinga, Porto Alegre" usando a função `ox.graph_from_place()`. Esta rede consiste em nós (interseções) e arestas (trechos de rua).

In [ ]:
# Instale a osmnx
!pip install osmnx -q

# Importe as bibliotecas
import osmnx as ox
import matplotlib.pyplot as plt



In [ ]:
# Defina o local para o qual queremos obter as informações do OSM
lugar="Restinga, Porto Alegre"

# Extraia a rede de ruas com base no nome da localização
rede=ox.graph_from_place(lugar)



### Visualizando a Rede de Ruas

Utilizamos `ox.plot_graph()` para visualizar a rede de ruas recém-extraída. Este plot mostra os nós como pontos brancos e as arestas como linhas brancas, sobre um fundo escuro padrão. É uma forma rápida de verificar se a extração foi bem-sucedida e se a área geográfica está correta.

In [ ]:
# Plote as arestas e nós que representam o arruamento do bairro
fig,ax=ox.plot_graph(rede)

### Separando Nós e Arestas em GeoDataFrames

Para manipulações mais detalhadas, é útil separar a rede (graph) em dois GeoDataFrames distintos: um para os nós (`nos`) e outro para as arestas (`arestas`). A função `ox.graph_to_gdfs()` faz exatamente isso, facilitando a análise e visualização de cada componente separadamente.

In [ ]:
# Extraindo e separando nós e arestas em dois geodataframes
nos,arestas= ox.graph_to_gdfs(rede)

### Visualização Individual de Nós e Arestas

Podemos plotar os `nos` e as `arestas` separadamente para entender suas distribuições. Os nós representam pontos como interseções ou extremidades de ruas, enquanto as arestas representam os segmentos de rua.

In [ ]:
# Visualizando apenas os nós
nos.plot()

In [ ]:
# Visualizando apenas as arestas
arestas.plot()

### Plot Combinado: Limite do Bairro e Rede de Ruas

Nesta seção, criamos um plot mais informativo, combinando o limite do bairro (`restinga`) com a rede de ruas (`arestas`). Definimos o limite do bairro com uma `facecolor` preta e as ruas em branco para um contraste claro. Isso ajuda a visualizar a rede de ruas dentro do contexto geográfico do bairro.

In [ ]:
# Crie a figura, o eixo de plotagem e define o tamanho da figura
fig, eixo= plt.subplots(figsize=(10,10))

# Plote cada um dos geodataframes no eixo de plotagem para garantir que estejam todos na mesma figura
restinga.plot(ax=eixo,facecolor="gray")
arestas.plot(ax=eixo, linewidth=1, edgecolor="white")

### Extraindo Polígonos de Edifícios (Prédios) do OSM

Para obter informações sobre os edifícios no bairro, usamos a função `ox.features_from_polygon()`. Primeiro, obtemos o polígono da área (`place_polygon`) a partir do nome do lugar, e então extraímos as feições com a tag `building=True` dentro desse polígono. Isso nos fornece um GeoDataFrame (`predios`) contendo a geometria de todos os edifícios mapeados no OpenStreetMap para a Restinga.

In [ ]:
# Obtenha os polígonos dos prédios do bairro e armazene na variável predios. As tags definem os tipos
# de objetos que serão buscados. Você pode obter todas as possíveis tags no manual do osmnx
predios=ox.features_from_place(lugar, tags={"building":True})

# Mostre o tipo da variável predios
print(type(predios))

# Plote os predios
predios.plot()

In [ ]:
predios

In [ ]:
# importa a biblioteca folium
import folium


In [ ]:
#cria a variavel map_restinga
map_restinga = folium.Map(location=[-30.16,-51.14], zoom_start = 17)
#manda desenhar o map
map_restinga

In [ ]:
#adiciona a variável predios ao map
folium.GeoJson(data=predios["geometry"]).add_to(map_restinga)
#manda desenhar o map
map_restinga

In [ ]:
#adiciona a variável arestas ao map
folium.GeoJson(data=arestas["geometry"]).add_to(map_restinga)
#manda desenhar o map
map_restinga



In [ ]:
arestas.columns

In [ ]:
arestas.head(5)

In [ ]:
#cria um caminho para salvar a saida
#manda a consulta para um arquivo shape

out1='/content/drive/MyDrive/shapes/restinga_ruas.shp'

#arestas['maxspeed'] = arestas['maxspeed'].astype(str)
arestas['name'] = arestas['name'].astype(str)
#arestas['reversed'] = arestas['reversed'].astype(str)
#arestas['oneway'] = arestas['oneway'].astype(str)
#arestas['lanes'] = arestas['lanes'].astype(str)
arestas.columns= [c.replace(",", " ") for c in list(arestas.columns)]

ruas=arestas[['name','geometry']]

ruas.to_file(out1)